# 03 — MLflow Experiment Tracking

End-to-end run of the BIC sweep with full MLflow logging.

**What this notebook does:**
1. Fetches S&P 500 OHLCV data and builds features
2. Calls `run_experiment()` — runs BIC sweep and logs everything to MLflow
3. Inspects the logged metrics and picks the best model
4. Loads the model back from the MLflow artifact store (`load_model_from_run`)
5. Shows how to open the MLflow UI

**Pre-requisites:** activate the venv (`source ~/Code/ml-venv/bin/activate`) and run from the repo root.

In [ ]:
import sys
sys.path.insert(0, '..')  # repo root when running from notebooks/

import logging
import yaml
import mlflow

from src.data.fetch import fetch_ohlcv
from src.features.engineer import build_feature_matrix
from src.model.experiment import run_experiment, load_model_from_run

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s — %(message)s')

with open('../configs/config.yaml') as f:
    config = yaml.safe_load(f)

print('Config loaded:')
for section, values in config.items():
    print(f'  {section}: {values}')

In [ ]:
# ── Fetch OHLCV and build feature matrix ─────────────────────────────────
print(f'Fetching {config["data"]["ticker"]} from {config["data"]["start_date"]} ...')
df = fetch_ohlcv(
    config['data']['ticker'],
    start=config['data']['start_date'],
    end=config['data']['end_date'],
)
print(f'OHLCV shape: {df.shape}  ({df.index[0].date()} → {df.index[-1].date()})')

features = build_feature_matrix(df, config)
print(f'Feature matrix shape: {features.shape}  columns: {features.columns.tolist()}')
features.tail(3)

In [ ]:
# ── Run the BIC sweep + log everything to MLflow ─────────────────────────
# This trains models for each k in config['model']['n_states_range'],
# selects the lowest BIC, and logs params / metrics / artifacts.
results = run_experiment(features, config, run_name='notebook_03_sweep')

print()
print(f"Run ID      : {results['run_id']}")
print(f"Best k      : {results['best_n_states']}")
print(f"Artifact URI: {results['artifact_uri']}")

print()
print('BIC scores per k:')
for k, bic in sorted(results['bic_scores'].items()):
    marker = '  ← best' if k == results['best_n_states'] else ''
    print(f'  k={k}  BIC={bic:,.2f}{marker}')

print()
print('AIC scores per k:')
for k, aic in sorted(results['aic_scores'].items()):
    marker = '  ← best' if k == results['best_n_states'] else ''
    print(f'  k={k}  AIC={aic:,.2f}{marker}')

In [ ]:
# ── Inspect the logged run via MLflow client ──────────────────────────────
mlflow.set_tracking_uri(config['mlflow']['tracking_uri'])
client = mlflow.tracking.MlflowClient()
run = client.get_run(results['run_id'])

print('Logged params:')
for k, v in sorted(run.data.params.items()):
    print(f'  {k}: {v}')

print()
print('Key scalar metrics:')
key_metrics = ['best_n_states', 'best_bic', 'best_aic', 'best_total_log_likelihood', 'n_training_samples']
for m in key_metrics:
    val = run.data.metrics.get(m, 'n/a')
    print(f'  {m}: {val}')

print()
print('Artifacts:')
for art in client.list_artifacts(results['run_id']):
    print(f'  {art.path}/')

In [ ]:
# ── Round-trip: load the model back from the MLflow artifact store ────────
model = load_model_from_run(results['run_id'], config)

print(f'Loaded model: {type(model).__name__}')
print(f'  n_components : {model.n_components}')
print(f'  covariance   : {model.covariance_type}')
print()
print('Emission means (rows = states, cols = [log_return, realized_vol, volume_ratio]):')
import pandas as pd
means_df = pd.DataFrame(
    model.means_,
    columns=['log_return', 'realized_vol', 'volume_ratio'],
    index=[f'State {i}' for i in range(model.n_components)]
)
means_df['ann_return_%'] = means_df['log_return'] * 252 * 100
means_df['ann_vol_%']    = means_df['realized_vol'] * 100
print(means_df.round(6).to_string())

print()
print('To view all runs in the MLflow UI run:')
print(f'  mlflow ui --backend-store-uri {config["mlflow"]["tracking_uri"]} --port 5000')
print('Then open: http://localhost:5000')